# 05 · Latents with exege: regions, PCA and a sparse autoencoder

[exege](https://github.com/mahf708/exege) reads a model's internal activations as a **latent archive**: one array per layer, `(times, nodes, channels)`, on a grid. Here each node is one of the 4050 patch tokens and the channels are the 384 residual-stream dimensions.

This notebook exports a small archive from the ViT, then asks exege:

* which channels respond in a region;
* where else the model "looks the same" (cosine similarity);
* what a sparse autoencoder finds.

In [ ]:
%matplotlib inline
import sys, os
from pathlib import Path
try:
    import vit_interp
except ImportError:  # not installed: use this checkout, $VIT_INTERP_REPO, or the shared one
    for repo in (Path.cwd().parent, Path(os.environ.get("VIT_INTERP_REPO", "/home/ac.ngmahfouz/vit-interp"))):
        if (repo / "vit_interp").is_dir():
            sys.path.insert(0, str(repo)); break
import numpy as np, torch, matplotlib.pyplot as plt
from vit_interp import playground as pg
em, meta = pg.load()      # or pg.load("/path/to/checkpoint.pt"); $PLAYGROUND_CKPT also works
FAST = pg.FAST            # PLAYGROUND_FAST=1 -> small settings (CPU tests)

In [ ]:
import subprocess, shutil
REPO = Path(pg.__file__).resolve().parents[1]
EXEGE = str(Path(sys.executable).with_name("exege"))
OUT = Path("/lcrc/globalscratch") / os.environ["USER"] / "vit-interp-play" / "latents-2010-jan"
def run(*cmd):
    r = subprocess.run([str(c) for c in cmd], cwd=REPO, capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-2000:]); r.check_returncode()
ckpt = os.environ.get("PLAYGROUND_CKPT") or pg.default_checkpoint()
n = 12 if FAST else 40
run(sys.executable, "-m", "vit_interp.export", ckpt, OUT, "--n", n, "--layers", "1,4,7", "--embed", "--overwrite")

In [ ]:
run(EXEGE, "latents", "info", OUT)

## A region: the North Atlantic storm track
Rank the channels that respond most within 1500 km of (50°N, 30°W) and map where else the model looks like this.

In [ ]:
from exege.latents import Region, analyze_region, open_source
source = open_source(str(OUT))
layer = source.info().layers[-1].index          # the last block in the archive
res = analyze_region(source, time=0, layer=layer, region=Region(lat=50, lon=-30, radius_km=1500),
                     centered=True, n_components=3)
print("top channels:", res.ranking.channels[:10])
grid = source.grid()
pg.plot_maps({"cosine similarity to the region": grid.to_map(res.similarity),
              "PC 1 fitted in the region": grid.to_map(res.scores[:, 0])}, cols=2);

The same from the command line, with a reproducible JSON summary:

In [ ]:
run(EXEGE, "latents", "region", OUT, "--lat", 50, "--lon", -30, "--radius-km", 1500, "--centered", "--top", 6, "--pcs", 2)

## A sparse autoencoder on one layer
This one is tiny and quick, to see the workflow. For real features, export more times and train longer on a GPU.

In [ ]:
(OUT / "bases").mkdir(exist_ok=True)
run(EXEGE, "nn", "sae", OUT, "--layer", layer, "--features", 128 if FAST else 512, "--k", 16,
    "--epochs", 1 if FAST else 3, "--out", OUT / "bases" / "sae-last.npz")
run(EXEGE, "latents", "region", OUT, "--lat", 50, "--lon", -30, "--radius-km", 1500,
    "--basis", OUT / "bases" / "sae-last.npz", "--features", 3)

## Try this

1. Launch the app: `exege app --latents <OUT parent>` (on a compute node, then tunnel the port as in the README). Click around regions and layers.
2. Compare layers: is the North Atlantic "similar to" the North Pacific at layer 1 (geography-driven) and less so at layer 7 (weather-driven)?
3. Export a **month** (`--n 124`) and train an SAE on a GPU (`--device cuda --epochs 10`). Find a feature that tracks fronts, the ITCZ or sea ice.
4. Correlate a feature with a physical field: `exege latents fields OUT --basis ... --feature N`. See `exege latents --help`.
5. Hugging Face: archives can be read straight from `hf://datasets/<owner>/<repo>/<folder>`. See the exege docs.